# 21. 统计计算与随机抽样

<!-- module-learning-arc:start -->
> **NumPy 模块主线｜第 6 / 6 步：用统计与抽样形成判断**
>
> **持续应用背景：** 为区域仓库建立补货预警矩阵：把门店、商品、库存和需求组织成数组，逐步完成定位、广播计算、排序和抽样复核。
>
> **承接上一阶段：** 向量化与广播  →  **本章任务：** 统计计算与随机抽样  →  **下一步：** 模块大作业《连锁门店补货预警矩阵》
>
> **大作业连接：** 本章练习将成为《连锁门店补货预警矩阵》的一部分，最终需要把数组建模、风险筛选、广播计算和抽样复核组合成一份可执行的补货清单。
<!-- module-learning-arc:end -->


## 本章场景

平时做数据分析，拿到一批销售数字或实验数据，最常问的就是：这批数整体大概是多少、波动大不大、哪些值明显偏高或偏低。



## 本章目标

学完本章，你将能够：

- **理解**：理解常用的统计量（均值/中位数/分位/样本）与随机抽样。
- **操作**：能用 numpy 计算统计量并抽样。
- **迁移**：能用统计数字描述一批经营数据（总额、均值、分位、波动）。


## 21.1 核心概念

**背景引入**：平时做数据分析，拿到一批销售数字或实验数据，最常问的就是：这批数整体大概是多少、波动大不大、哪些值明显偏高或偏低。数组统计（求和、均值、分位数）与随机抽样正是回答这些问题的基本功，既能看清一批数的整体水平，也能用可复现的方式模拟「再抽一批看看会怎样」。本节把这两项一起讲清楚。

- axis=0聚合行并保留列，axis=1聚合列并保留行。
- 均值要结合标准差和分位数解释。
- 使用default_rng和固定种子保证结果可复现。

**口诀**：看整体水平用均值，看波动用标准差，怕极端值被带偏就补个中位数和分位。


## 21.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| sum()、mean() 和 median() | `np.array()`、`values.sum()`、`values.mean()`、`np.median()` | 均值和中位数要结合分布与异常值一起解释。 | 不说明axis导致统计口径错误 |
| axis 聚合 | `np.array()`、`sales.sum()`、`sum()` | axis=0按列聚合，axis=1按行聚合，使用前先确认业务维度。 | 只报告均值忽略分布 |
| quantile() | `np.array()`、`np.quantile()` | 分位数可以描述分布位置，并帮助构造异常阈值。 | 每次运行使用不同随机状态导致结果无法复现 |
| default_rng() | `np.random.default_rng()`、`rng.integers()`、`np.random.RandomState()`、`rng.randint()` | 固定种子可以让练习结果可复现；旧版NumPy使用RandomState作为兼容替代。 | 不说明axis导致统计口径错误 |
| choice() | `np.random.default_rng()`、`np.random.RandomState()`、`np.arange()`、`rng.choice()` | choice可以抽取样本，replace=False表示不重复。 | 只报告均值忽略分布 |
| normal() 和 poisson() | `np.random.default_rng()`、`np.random.RandomState()`、`rng.normal()`、`rng.poisson()` | 不同分布适合模拟不同类型的随机现象，参数要写清楚。 | 每次运行使用不同随机状态导致结果无法复现 |


## 21.3 示例 1：按轴统计

先确认每个轴代表什么业务维度。

**背景引入**：一张“区域×月份”的销售表，既要算“每个月全区域合计”，又要算“每个区域平均”，还得知道整体波动。同样是求和、求平均，按行还是按列结果完全不同，axis 决定了“沿哪个方向动”。

**讲解**：axis=0 沿行方向聚合、保留列（得到“每月合计”）；axis=1 沿列方向聚合、保留行（得到“每区平均”）；median/std 不带 axis 就是对整张表整体算。

- sales.sum(axis=0)：沿第 0 轴（行）方向加总，剩下每一列 → 得到每月合计；
- sales.mean(axis=1)：沿列方向算平均，剩下每一行 → 得到每区平均；
- np.median(sales)、sales.std() 不带 axis，对整张表所有元素求中位数、标准差；
- **口诀**：axis=0 收行留列、axis=1 收列留行；先想清每行每列是啥业务，再选方向。


<!-- math-foundation:chapter-21 -->
### 数学推导｜均值、方差与标准误

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜均值确定数据中心。** $\bar{x}$ 让正负离差相互抵消：

$$
\sum_{i=1}^{n}(x_i-\bar{x})=0
$$

**第 2 步｜平方离差衡量离散。** 样本方差用 $n-1$ 校正估计偏差，再令 $s=\sqrt{s^2}$ 得到标准差。

**第 3 步｜均值比单个观测更稳定。** 若样本近似独立同分布，

$$
\operatorname{Var}(\bar{X})=\frac{\sigma^2}{n}
\quad\Longrightarrow\quad
SE(\bar{X})\approx\frac{s}{\sqrt{n}}
$$

样本量扩大 4 倍，均值标准误约缩小一半，而不是缩小到四分之一。

**把上面的关系收束为本章计算式：**

$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i,\qquad s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2,\qquad SE=\frac{s}{\sqrt{n}}
$$

**符号解释：** $\bar{x}$ 是样本均值，$s^2$ 是样本方差，$SE$ 是均值标准误。

**代码对应：** `np.mean(x)`、`np.var(x, ddof=1)`、`np.std(x, ddof=1) / np.sqrt(len(x))`。

**使用边界：** 随机抽样要固定种子并说明抽样总体；标准误不是原始数据的标准差。


In [ ]:
import numpy as np

sales = np.array(
    [[120, 150, 180, 210], [98, 132, 145, 170], [110, 128, 160, 188]]
)
print("每月合计:", sales.sum(axis=0))
print("每区平均:", sales.mean(axis=1))
print("整体中位数:", np.median(sales))
print("标准差:", sales.std().round(2))


## 21.4 示例 2：分位数与异常阈值

IQR规则是识别潜在异常的启发式方法。

**背景引入**：核查一批销售或支出数据时，光看平均值会漏掉异常——比如有一笔 120 明显偏高。用分位数描述“分布位置”，再按 IQR 规则算出一个阈值，就能把潜在的异常自动揪出来。

**讲解**：np.quantile(values, [0.25, 0.5, 0.75]) 一次算出下四分位 Q1、中位数、上四分位 Q3；IQR = Q3 - Q1，upper = Q3 + 1.5 × IQR 作为上界，超出的即为潜在异常。

- q1, median, q3 一次取 25%、50%、75% 三个分位点，概括分布位置；
- iqr = q3 - q1 是四分位距，衡量中间一半数据的离散程度；
- upper = q3 + 1.5 * iqr，配合 values > upper 布尔筛选，把超上界的 120 标出来；
- **口诀**：Q1、中位、Q3 描述位置，IQR 量中段宽窄，上界之外就是潜在异常。


In [ ]:
values = np.array([52, 58, 61, 63, 65, 68, 72, 120])
q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(q1, median, q3)
print("潜在异常:", values[values > upper])


## 21.5 示例 3：随机抽样与模拟

同一种子产生相同序列，便于复现实验。

**背景引入**：做运营模拟——从 100 到 200 的单量里抽 8 天核对，或模拟未来 30 天的日均订单。随机数若每次运行都变，结果就没办法对账；用固定种子生成器，同一次“随机”可复现。

**讲解**：np.random.default_rng(15) 创建一个带固定种子的随机数生成器；rng.choice 从指定范围不重复抽样，rng.poisson 生成泊松分布的随机订单数，同一种子序列固定可复现。

- rng = np.random.default_rng(15)：固定种子 15，每次生成的随机序列都相同；
- rng.choice(np.arange(100, 201), size=8, replace=False)：从 100~200 不重复抽 8 个；
- rng.poisson(lam=24, size=30)：模拟日均 24 单的 30 天订单数，再 .mean() 求模拟均值；
- **口诀**：default_rng 配固定种子，随机才可复现；choice 不重复抽样，poisson 模拟订单流。


In [ ]:
rng = np.random.default_rng(15)
sample = rng.choice(np.arange(100, 201), size=8, replace=False)
simulated_orders = rng.poisson(lam=24, size=30)
print("抽样:", sample)
print("模拟日均订单:", simulated_orders.mean().round(2))


## 21.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# sum()、mean() 和 median()
# 均值和中位数要结合分布与异常值一起解释。
import numpy as np

values = np.array([10, 12, 13, 15, 100])
print("总和:", values.sum())
print("均值:", values.mean())
print("中位数:", np.median(values))


In [ ]:
# axis 聚合
# axis=0按列聚合，axis=1按行聚合，使用前先确认业务维度。
import numpy as np

sales = np.array([[100, 120, 140], [80, 90, 110]])
print("每月:", sales.sum(axis=0))
print("每地区:", sales.sum(axis=1))


In [ ]:
# quantile()
# 分位数可以描述分布位置，并帮助构造异常阈值。
import numpy as np

values = np.array([10, 12, 13, 15, 18, 20, 100])
q1, q3 = np.quantile(values, [0.25, 0.75])
print(q1, q3)


In [ ]:
# default_rng()
# 固定种子可以让练习结果可复现；旧版NumPy使用RandomState作为兼容替代。
import numpy as np

if hasattr(np.random, "default_rng"):
    rng = np.random.default_rng(42)
    values = rng.integers(1, 11, size=5)
else:
    rng = np.random.RandomState(42)
    values = rng.randint(1, 11, size=5)
print(values)


In [ ]:
# choice()
# choice可以抽取样本，replace=False表示不重复。
import numpy as np

if hasattr(np.random, "default_rng"):
    rng = np.random.default_rng(7)
else:
    rng = np.random.RandomState(7)
population = np.arange(1, 21)
print(rng.choice(population, size=5, replace=False))


In [ ]:
# normal() 和 poisson()
# 不同分布适合模拟不同类型的随机现象，参数要写清楚。
import numpy as np

if hasattr(np.random, "default_rng"):
    rng = np.random.default_rng(9)
else:
    rng = np.random.RandomState(9)
print("正态样本:", rng.normal(loc=100, scale=10, size=4).round(2))
print("泊松样本:", rng.poisson(lam=12, size=4))


### 练一练 18.6：给一周销量算账并抽样

用本节学到的 **求和 / 均值 / 分位数** 和 **随机抽样** 完成一个小任务：某店一周 7 天的销量是 `[120, 135, 128, 142, 131, 150, 118]`。

1. 定义数组 `week_sales`（上面的 7 个数字）。
2. 算出一周的 **总和**、**均值** 和 **90% 分位数**（均值保留 2 位小数）。
3. 用固定种子 `np.random.default_rng(2024)` 从这 7 天里 **不重复抽取 3 天** 作为检查样本。

先在练习单元格里填好你的版本，运行前先想清楚结果大概是多少；需要参考时再回看紧邻的示例。


In [ ]:
# 请在下方填写代码
import numpy as np

# --- 步骤 1：定义一周 7 天销量 ---
# --- 步骤 2：算账，把空填上 ---
# --- 步骤 3：固定种子，不重复抽取 3 天 ---


In [ ]:
# 完整答案
import numpy as np

week_sales = np.array([120, 135, 128, 142, 131, 150, 118])

total = week_sales.sum()
average = week_sales.mean()
p90 = np.quantile(week_sales, 0.90)

rng = np.random.default_rng(2024)
inspection = rng.choice(week_sales, size=3, replace=False)

print("总和:", total)
print("均值:", round(average, 2))
print("90% 分位:", round(p90, 2))
print("检查样本:", inspection)


**输出解读**：`week_sales.sum()` 是 `924`；`week_sales.mean()` 约为 `132.00`（保留 2 位）；`np.quantile(week_sales, 0.90)` 约为 `145.2`（在 142 与 150 之间按位置插值，而不是简单取某一个值）；使用 `default_rng(2024)` 后，抽出的 3 天在**每次运行都相同**——固定种子就是为了让"随机"变得可复现。要点：看一批数不能只看均值，还要结合标准差与分位数判断分布。


## 21.7 独立迁移练习

先预测 shape，再修改一个数组或筛选条件，解释结果变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 21.8 本章实训：axis与布尔筛选

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np

matrix = np.arange(1, 13).reshape(3, 4)
print("原数组：\n", matrix)
print("每行合计：", matrix.sum(axis=1))
print("每列合计：", matrix.sum(axis=0))


### 21.8.1 第一个结果怎么读

`axis=1` 保留行，沿列方向计算；`axis=0` 保留列，沿行方向计算。先看 shape，再解释结果长度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
even = matrix[matrix % 2 == 0]
print("偶数：", even)
print("偶数数量：", even.size)
print("偶数平均值：", even.mean())


### 21.8.2 第二个结果怎么读

第二个实验不改原数组，而是用布尔条件筛选新数组。请思考：如果条件改成 `matrix > 8`，输出会怎样变化？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 21.9 错误恢复：数组形状不匹配怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import numpy as np

matrix = np.arange(6).reshape(2, 3)
try:
    result = matrix + np.array([10, 20])
except ValueError as error:
    print("形状问题：", type(error).__name__)
    result = matrix + np.array([10, 20, 30])
print("修复后的结果：")
print(result)


### 21.9.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先看两个数组的 shape，再判断能否广播。修复不是随意 reshape，而是让数据结构和业务含义一致。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 21.10 易错点提醒

- 不说明axis导致统计口径错误
- 只报告均值忽略分布
- 每次运行使用不同随机状态导致结果无法复现


## 21.11 练习与作业

1. 模拟60天销售额
2. 计算均值、中位数和90%分位数
3. 抽取5天作为检查样本

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 21.12 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“模拟60天销售额”。
2. **独立完成**：不复制示例代码，完成“计算均值、中位数和90%分位数”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“抽取5天作为检查样本”，用一两句话说明你修改了什么。

### 21.12.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 21.12.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import numpy as np

# TODO: 模拟60天销售额（正态分布，均值180，标准差35，最小值60）
# TODO: 抽取5天样本
# TODO：请在下方完成 —— 18.12 练习与作业 1. 模拟60天销售额 2. 计算均值、中位数和90%分位数 3. 抽取5天作为检查样本 提交前


In [ ]:
import numpy as np

rng = np.random.default_rng(150)
daily_sales = rng.normal(180, 35, 60).clip(60)
inspection = rng.choice(daily_sales, size=5, replace=False)
print("均值:", daily_sales.mean().round(2))
print("中位数:", np.median(daily_sales).round(2))
print("90%分位:", np.quantile(daily_sales, 0.9).round(2))
print("检查样本:", inspection.round(2))


## 21.13 小结

使用axis完成数组统计，并通过现代随机数生成器进行可复现抽样。

**迁移思考**：

1. 如果一个 4×5 数组按 axis=0 求和，结果是什么形状？按 axis=1 呢？
2. 为什么随机模拟需要固定种子？什么场景下应该使用不同的种子？



### 21.13.1 你已经掌握

- 按轴聚合
- 计算分位数和离散程度
- 创建可复现随机数
- 执行抽样、洗牌和模拟


### 21.13.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 21.13.3 需要注意

- 不说明axis导致统计口径错误
- 只报告均值忽略分布
- 每次运行使用不同随机状态导致结果无法复现


### 21.13.4 完成检查

- [ ] 能够按轴聚合
- [ ] 能够计算分位数和离散程度
- [ ] 能够创建可复现随机数
- [ ] 能够执行抽样、洗牌和模拟


### 21.13.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
